In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
import torch.optim as optim
from torch.utils.data import Dataset , DataLoader

In [2]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [3]:
df=pd.read_csv("/content/fashion-mnist_train.csv")
df=df.dropna()

In [4]:
torch.manual_seed(42)

In [5]:
df

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,9,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,6,0,0,0,0,0,0,0,5,0,...,0.0,0.0,0.0,30.0,43.0,0.0,0.0,0.0,0.0,0.0
3,0,0,0,0,1,2,0,0,0,0,...,3.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
4,3,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5198,0,0,0,0,0,0,0,1,1,0,...,187.0,171.0,164.0,1.0,0.0,2.0,0.0,0.0,0.0,0.0
5199,5,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5200,8,0,0,0,0,0,0,0,0,0,...,191.0,194.0,191.0,197.0,210.0,215.0,0.0,0.0,0.0,0.0
5201,1,0,0,0,0,0,0,0,0,0,...,69.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [6]:
x = df.iloc[:, 1:].values
y = df.iloc[:, 0].values

In [7]:
x_train , x_test ,y_train, y_test = train_test_split(x,y,test_size=0.2, random_state=42)

In [8]:
x_train=x_train/255.0
x_test=x_test/255.0


In [9]:
class MyNN(nn.Module):
    def __init__(self, input_dim, output_dim, num_hidden_layer, num_neuron , dropouts):
        super().__init__()

        layers = []

        for i in range(num_hidden_layer):
            layers.append(nn.Linear(input_dim, num_neuron))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropouts))
            input_dim = num_neuron

        layers.append(nn.Linear(input_dim, output_dim))

        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

In [10]:
def objective(trial):
    # Hyperparameters
    num_hidden_layer = trial.suggest_int("hidden_layers", 1, 5)
    num_neuron = trial.suggest_int("neuron", 10, 128, step=10)
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)
    epochs = trial.suggest_int("epochs", 10, 50)
    dropouts=trial.suggest_float("dropouts",0.0,0.5)



    input_dim = x_train.shape[1]
    output_dim = 10

    # Model
    model = MyNN(input_dim, output_dim, num_hidden_layer, num_neuron, dropouts)
    model = model.to(device)

    # Loss + Optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)


    # ========== Training ==========
    for epoch in range(epochs):
        model.train()
        for batch_features, batch_labels in train_loader:
            batch_features = batch_features.to(device)
            batch_labels = batch_labels.to(device)

            outputs = model(batch_features)
            loss = criterion(outputs, batch_labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    # ========== Evaluation ==========
    model.eval()
    total = 0
    correct = 0

    with torch.no_grad():
        for batch_features, batch_labels in test_loader:
            batch_features = batch_features.to(device)
            batch_labels = batch_labels.to(device)

            outputs = model(batch_features)
            _, predicted = torch.max(outputs, 1)

            total += batch_labels.shape[0]
            correct += (predicted == batch_labels).sum().item()

    accuracy = correct / total
    return accuracy

In [11]:
# create CustomDataset Class
class CustomDataset(Dataset):

  def __init__(self, features, labels):

    self.features = torch.tensor(features, dtype=torch.float32)
    self.labels = torch.tensor(labels, dtype=torch.long)

  def __len__(self):

    return len(self.features)

  def __getitem__(self, index):

    return self.features[index], self.labels[index]




In [12]:
train_dataset=CustomDataset(x_train,y_train)
test_dataset=CustomDataset(x_test,y_test)


In [13]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [14]:
! pip install optuna


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 28.2 MB/s eta 0:00:00


In [18]:
import optuna
study=optuna.create_study(direction="maximize")
study.optimize(objective,n_trials=20)

[I 2026-09-24 04:53:16,756] A new study created in memory with name: no-name-dfdbc58d-bfcc-4d09-9f0a-1f4959496fe8
/tmp/ipykernel_1044/2488562101.py:4: UserWarning: The distribution is specified by [10, 128] and step=10, but the range is not divisible by `step`. It will be replaced with [10, 120].
  num_neuron = trial.suggest_int("neuron", 10, 128, step=10)
[I 2026-09-24 04:53:28,882] Trial 0 finished with value: 0.7089337175792507 and parameters: {'hidden_layers': 5, 'neuron': 80, 'learning_rate': 0.0035973086056119395, 'epochs': 33, 'dropouts': 0.4597999725994888}. Best is trial 0 with value: 0.7089337175792507.
[I 2026-09-24 04:53:34,397] Trial 1 finished with value: 0.8453410182516811 and parameters: {'hidden_layers': 2, 'neuron': 60, 'learning_rate': 0.0020452645450513424, 'epochs': 22, 'dropouts': 0.07160657863314884}. Best is trial 1 with value: 0.8453410182516811.
[I 2026-09-24 04:53:42,403] Trial 2 finished with value: 0.8213256484149856 and parameters: {'hidden_layers': 4, 'ne

In [19]:

 study.best_params

{'hidden_layers': 3,
 'neuron': 110,
 'learning_rate': 0.0006664473869455701,
 'epochs': 37,
 'dropouts': 0.42255039157106333}

In [21]:
study.best_value

0.851104707012488

In [23]:
Final_model =MyNN(
    input_dim=x_train.shape[1],
    output_dim=10,
    num_hidden_layer=study.best_params["hidden_layers"],
    num_neuron=study.best_params["neuron"],
    dropouts=study.best_params["dropouts"]
)
Final_model = Final_model.to(device)

In [37]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(Final_model.parameters(), lr=0.00019281977229289828)
for epoch in range(study.best_params['epochs']):
    Final_model.train()
    for batch_features, batch_labels in train_loader:
        batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
        outputs = Final_model(batch_features)
        loss = criterion(outputs, batch_labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1} done")
    Final_model.eval()
    with torch.no_grad():
        total = 0
        correct = 0
        for batch_features, batch_labels in test_loader:
            batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
            outputs = Final_model(batch_features)
            _, predicted = torch.max(outputs, 1)
            total += batch_labels.shape[0]
            correct += (predicted == batch_labels).sum().item()
        print(f"Accuracy: {correct/total}")


Epoch 1 done
Accuracy: 0.8491834774255523
Epoch 2 done
Accuracy: 0.8443804034582133
Epoch 3 done
Accuracy: 0.8424591738712777
Epoch 4 done
Accuracy: 0.8434197886647454
Epoch 5 done
Accuracy: 0.8491834774255523
Epoch 6 done
Accuracy: 0.8463016330451489
Epoch 7 done
Accuracy: 0.8501440922190202
Epoch 8 done
Accuracy: 0.8549471661863592
Epoch 9 done
Accuracy: 0.8530259365994236
Epoch 10 done
Accuracy: 0.840537944284342
Epoch 11 done
Accuracy: 0.8453410182516811
Epoch 12 done
Accuracy: 0.8501440922190202
Epoch 13 done
Accuracy: 0.8434197886647454
Epoch 14 done
Accuracy: 0.8463016330451489
Epoch 15 done
Accuracy: 0.8472622478386167
Epoch 16 done
Accuracy: 0.851104707012488
Epoch 17 done
Accuracy: 0.851104707012488
Epoch 18 done
Accuracy: 0.8482228626320846
Epoch 19 done
Accuracy: 0.8491834774255523
Epoch 20 done
Accuracy: 0.8472622478386167
Epoch 21 done
Accuracy: 0.8386167146974063
Epoch 22 done
Accuracy: 0.8578290105667628
Epoch 23 done
Accuracy: 0.8463016330451489
Epoch 24 done
Accuracy: